## NN Pricer for 1D American Put + Binomial Tree Method

### Simulating GBM (as described in Chapter 4 in the report but for 1 underlying)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import time
import pandas as pd

np.random.seed(7)
torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

#Simulating GBM (as described in Chapter 4 in the report but for 1 underlying)(uses the Chapter 1 Ito derivation)
def simulate_gbm(S0, N, r, q, sigma, T, n_dates,
                 device = device, dtype = dtype):

    dt = T/n_dates
    S = torch.empty((N,n_dates + 1), device = device, dtype = dtype)
    S[:,0] = S0

    for i in range(n_dates):
        z = torch.randn((N,), device = device, dtype = dtype)

        S[:,i+1] =  S[:,i] * torch.exp((r-q-0.5*sigma**2)*dt + sigma*torch.sqrt(torch.tensor(dt, device = device, dtype = dtype))*z)

    return S

def payoff_put(S_t, K):
    return torch.relu(K - S_t)

### NN model and the backward recursion algorithm (Algorithm 1 in Chapter 1)

In [ ]:
#build the nn architecture

class NeuralNetwork(nn.Module):
    def __init__(self, d_input = 1):
        super().__init__()
        self.snn = nn.Sequential(
            torch.nn.Linear(d_input, 40),
            torch.nn.ReLU(),
            torch.nn.Linear(40, 1))
    def forward(self, X):
        z = self.snn(X).reshape(-1)
        return z
        
#train nn by minimising the MSE
def train_nn(X_scaled, y, epochs, lr, weight_decay, batch_size):

    nn_model = NeuralNetwork(d_input = 1).to(device = device, dtype = dtype)
    nn_model.train()

    optimiser = torch.optim.AdamW(nn_model.parameters(), lr = lr, weight_decay = weight_decay)

    loss_nn = nn.MSELoss()
    torch.optim.lr_scheduler.ExponentialLR(optimiser, gamma = 0.95)
    #torch.optim.lr_scheduler.StepLR(optimiser, step_size = 50, gamma = 0.5)         #tried different ones

    for epoch in range(epochs):

        indices = torch.randperm(X_scaled.shape[0], device = device)

        for i in range(0, X_scaled.shape[0], batch_size):
            idx = indices[i:i + batch_size]
            Xbatch = X_scaled[idx]
            ybatch = y[idx].reshape(-1)

            optimiser.zero_grad(set_to_none = True)
            train_pred = nn_model(Xbatch).reshape(-1)
            comp_loss = loss_nn(train_pred, ybatch)
            comp_loss.backward()
            optimiser.step()

    return nn_model

#use nn to estimate the continuation value

@torch.no_grad()                                                                      #https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html
def nn_pred(nn_model, X_scaled):
    nn_model.eval()
    return nn_model(X_scaled)


#the standard backward recursion algorithm
def nn_put_pricer(paths, N, K, T, n_dates, r,
                  epochs = 5, lr = 1e-2, weight_decay = 1e-8, batch_size = 1024):

    dt = T/n_dates

    v_cont = payoff_put(paths[:,n_dates], K = K)                            #value of the option at time T

    for i in range(n_dates - 1,0, -1):                                     #backwards recursion
        X_i = paths[:,i]
        y_i = v_cont

        #scaling

        Xi_scaled =  ((X_i - X_i.min()) / (X_i.max() - X_i.min()).clamp_min(1e-14)).reshape(-1,1)    #reshape to feed into the model as a tensor

        nn_model = train_nn(X_scaled = Xi_scaled, y = y_i, 
                            epochs = epochs, lr = lr, weight_decay = weight_decay, batch_size = batch_size)

        cont_est = nn_pred(nn_model,Xi_scaled)                                             #estimated continuation value
        c_hat = torch.exp(torch.tensor(-r * dt, device = device, dtype = dtype)) * cont_est    #estimated discounted continuation value


        g_i = payoff_put(X_i, K)                                    #payoff at time i
        exer = (g_i > c_hat) & (g_i > 0)                            #exercise if payoff > estimated continuation value and payoff>0

        v_cont = torch.where(exer, g_i, torch.exp(torch.tensor(-r * dt, device = device, dtype = dtype)) * v_cont)
                                                                     #update cashflow
                                                                     
    v0 = torch.exp(torch.tensor(-r*dt, device = device, dtype = dtype)) * v_cont.mean().item()
    
    return v0

### Compute option value + computational time

In [ ]:
S0 = 10.0              #spot price
T = 1.0                #maturity
K = 15.0               #strike price
n_dates = 10           #num of exercise dates
r = 0.06               #risk free interest rate
q = 0.1                #divident rate
sigma = 0.2            #volatility
N = 100000             #num of simulated paths

paths = simulate_gbm(S0 = S0, N = N, r = r, q = q, sigma = sigma,
                     T = T, n_dates = n_dates,
                     device = device, dtype = dtype)

start = time.time()

v = nn_put_pricer(paths = paths, N = N, K = K, T = T, n_dates = n_dates, r = r,
                  epochs = 5, lr = 1e-2, weight_decay = 1e-8, batch_size = 1024)
stop = time.time()


print( f"S0 = {S0}| {v:.3f} | time: {stop - start:.2f}s")

### Binomial Tree Method

In [ ]:
def BinomialPricer(S0, K, r, q, sigma, T, M):

    dt = T / M

    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp((r - q) * dt) - d)/(u - d)

    ST = np.array([S0 * (u**i) * (d**(M - i)) for i in range(M + 1)])

    V = np.maximum(K - ST, 0.0)

    for i in range(M - 1,-1, -1):
        ST = ST[1:] / u
        V = np.exp(-r * dt) * (p * V[1:] + (1 - p) * V[:-1])
        exer_value = np.maximum(K - ST, 0.0)
        V = np.maximum(V, exer_value)                     #at each node we simply pick the greater value of immediate exercise vs cont value

    return V[0]

### Compute option value + computational time

In [ ]:
start = time.time()

v_binn = BinomialPricer(S0 = S0, K = K, r = r, sigma = sigma, q = q, T = T, M = 4000)

stop = time.time()

print( f"S0 = {S0}| {v_binn:.3f} | time: {stop - start:.2f}s")